## Module 6: Evaluation and Metrices

In [5]:
# torchmetrics
# pip install torchmetrics
import torchmetrics

num_classes = 4
f1_macro = torchmetrics.F1Score(task = 'multiclass', num_classes = num_classes, average = 'macro')
f1_weighted = torchmetrics.F1Score(task = 'multiclass', num_classes = num_classes, average = 'weighted')

preds = torch.tensor([0, 1, 2, 2, 3, 0])
labels = torch.tensor([0, 1, 2, 3, 3, 1])
print(f'Macro F1: {f1_macro(preds, labels):.4f}')
print(f'Weighted F1: {f1_weighted(preds, labels):.4f}')

Macro F1: 0.6667
Weighted F1: 0.6667


In [6]:
# Computing ROC and AUC
torch.manual_seed(42)
n = 200
y_true_auc = torch.randint(0, 2, (n,)).float()
y_prob_auc = torch.sigmoid(torch.randn(n) + y_true_auc * 1.5)   # model is somewhat correct

auroc = torchmetrics.AUROC(task = 'binary')
print(f'torchmetrics AUC: {auroc(y_prob_auc, y_true_auc.int()):.4f}')

torchmetrics AUC: 0.8892


In [7]:
# Choosing Optimal Threshold
def find_optimal_threshold(fprs, tprs, thresholds, strategy = 'youden'):
    """
    Find the optimal classification threshold from ROC data.

    Strategies:
      'youden'   : maximize (TPR - FPR) — Youden's J statistic
      'f1'       : maximize F1 score (requires re-computing over thresholds)
    """
    if strategy == 'youden':
        j_scores = tprs - fprs  # Youden's J = TPR - FPR
        best_idx = np.argmax(j_scores)
        return thresholds[best_idx], tprs[best_idx], fprs[best_idx]

    raise ValueError(f'Unknown Strategy: {strategy}')

best_thresh, best_tpr, best_fpr = find_optimal_threshold(fprs, tprs, thresholds)
print(f'Optimal Threshold: {best_thresh:.3f}')
print(f'At this threshold: TPR = {best_tpr:.3f}, FPR = {best_fpr:.3f}')

NameError: name 'fprs' is not defined

In [9]:
# Regression Metrics
import torch
import torch.nn as nn

def regression_metrics(y_true : torch.Tensor, y_pred : torch.Tensor) -> dict:
    y_true = y_true.view(-1).float()
    y_pred = y_pred.view(-1).float()

    mae = (y_pred - y_true).abs().mean()
    mse = ((y_pred - y_true) ** 2).mean()
    rmse = torch.sqrt(mse)
    
    ss_res = ((y_pred - y_true) ** 2).sum()
    ss_tot = ((y_true - y_true.mean()) ** 2).sum()
    r2 = 1 - ss_res / (ss_tot + 1e-8)

    return {
        'mae' : mae.item(),
        'rmse' : rmse.item(),
        'r2' : r2.item()
    }

y_true = torch.tensor([3.0, -0.5, 2.0, 7.0])
y_pred = torch.tensor([2.5, 0.0, 2.0, 8.0])

metrics = regression_metrics(y_true, y_pred)
print(f'MAE: {metrics["mae"]:.4f}')
print(f'RMSE: {metrics["rmse"]:.4f}')
print(f'R2: {metrics["r2"]:.4f}')

MAE: 0.5000
RMSE: 0.6124
R2: 0.9486


In [11]:
# Using torchmetrics
import torchmetrics
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

accuracy = torchmetrics.Accuracy(task = 'binary').to(device)
precision = torchmetrics.Precision(task = 'binary').to(device)
recall = torchmetrics.Recall(task = 'binary').to(device)
f1 = torchmetrics.F1Score(task = 'binary').to(device)
auroc = torchmetrics.AUROC(task = 'binary').to(device)

batches = [
    (torch.tensor([1, 0, 1, 1]), torch.tensor([0.9, 0.1, 0.8, 0.7])),
    (torch.tensor([0, 1, 0, 0]), torch.tensor([0.2, 0.6, 0.3, 0.1])),
    (torch.tensor([1, 1, 0, 1]), torch.tensor([0.85, 0.95, 0.4, 0.75]))
]

for y_true_batch, y_prob_batch in batches:
    y_true_batch = y_true_batch.to(device)
    y_prob_batch = y_prob_batch.to(device)

    accuracy.update(y_prob_batch, y_true_batch)
    precision.update(y_prob_batch, y_true_batch)
    recall.update(y_prob_batch, y_true_batch)
    f1.update(y_prob_batch, y_true_batch)
    auroc.update(y_prob_batch, y_true_batch)

print(f'Accuracy: {accuracy.compute():.4f}')
print(f'Precision: {precision.compute():.4f}')
print(f'Recall: {recall.compute():.4f}')
print(f'F1: {f1.compute():.4f}')
print(f'AUROC: {auroc.compute():.4f}')

accuracy.reset()
precision.reset()
recall.reset()
f1.reset()
auroc.reset()

Accuracy: 1.0000
Precision: 1.0000
Recall: 1.0000
F1: 1.0000
AUROC: 1.0000


In [12]:
# Methods for Class Imbalance Problem

# 1. Class weights in loss fn = class 0 hhas 990 samples, class 1 has 10 samples
# weight class 1 as 99x more important
import torch.nn as nn
pos_weight = torch.tensor([99.0])
criterion = nn.BCEWithLogitsLoss(pos_weight = pos_weight)


# 2. Oversampling the minority class
from torch.utils.data import WeightedRandomSampler
class_counts = torch.tensor([99.0, 10.0])
weights_per_class = 1.0 / class_counts

sample_weights = weights_per_class[y_train]

sampler = WeightedRandomSampler(
    weights = sample_weights, num_samples = len(sample_weights),  replacement = True
)
train_loader = DataLoader(train_dataset, batch_size = 32, sampler = sampler)


# 3. Threshold Adjustment = Instead of default threshold of 0.5, finding optimal threshold on 
# validation set using ROC curve or F1 maximization.

NameError: name 'y_train' is not defined

In [13]:
# Full Evaluation Pipeline
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchmetrics

def evaluate_binary_classifier(
    model: nn.Module, loader: DataLoader, criterion: nn.Module, device: str = 'cpu', 
    threshold: float = 0.5) -> dict:
    
    model.eval()

    acc = torchmetrics.Accuracy(task = 'binary', threshold = threshold).to(device)
    prec = torchmetrics.Precision(task = 'binary', threshold = threshold).to(device)
    rec = torchmetrics.Recall(task = 'binary', threshold = threshold).to(device)
    f1 = torchmetrics.F1(task = 'binary', threshold = threshold).to(device)
    auroc = torchmetrics.AUROC(task = 'binary').to(device)

    total_loss = 0.0
    total_n = 0

    with torch.no_grad():
        for bX, by in loader:
            bX, by = bX.to(device), by.to(device)

            logits = model(bX)
            probs = torch.sigmoid(logits).squeeze(1)
            labels = by.squeeze(1).long()
            labels_float = by.squeeze(1).float()

            loss = criterion(logits.squeeze(1), lables_float)
            total_loss += loss.item() * bX.size(0)
            total_n += bX.size(0)

            acc.update(probs, labels)
            prec.update(probs, labels)
            rec.update(probs, labels)
            f1.update(probs, labels)
            auroc.update(probs, labels)

    return {
        'loss' : total_loss / total_n,
        'accuracy' : acc.compute().item(),
        'precision' : prec.compute().item(),
        'recall' : rec.compute().item(),
        'f1' : f1.compute().item(),
        'auc' : auroc.compute().item()
    }

def evaluate_multiclass_classifier(
    model: nn.Module, loader: DataLoader, criterion: nn.Module, num_classes: int, device: str = 'cpu') -> dict:
    
    model.eval()

    acc = torchmetrics.Accuracy(task = 'multiclass', num_classes = num_classes).to(device)
    f1_m = torchmetrics.F1Score(task = 'multiclass', num_classes = num_classes, average = 'macro').to(device)
    f1_w = torchmetrics.F1Score(task = 'multiclass', num_classes = num_classes, average = 'weighted').to(device)

    total_loss, total_n = 0.0, 0

    with torch.no_grad():
        for bX, by in loader:
            bX, by = bX.to(device), by.to(loader).long()

            logits = model(bX)
            loss = criterion(logits, by)
            preds = torch.argmax(logits, dim = 1)
            
            total_loss += loss.item() * bX.size(0)
            total_n += bX.size(0)

            acc.update(preds, by)
            f1_m.update(preds, by)
            f1_w.update(preds, by)

    return {
        'loss' : total_loss / total_n,
        'accuracy' : acc.compute().item(),
        'f1_macro' : f1_m.compute().item(),
        'f1_weigted' : f1_w.compute().item()
    }

def evaluate_regressor(
    model: nn.Module, loader: DataLoader, device: str = 'cpu', y_std: float = 1.0) -> dict:

    model.eval()

    mae_metric = torchmetrics.MeanAbsoluteError().to(device)
    mse_metric = torchmetrics.MeanSquaredError().to(device)

    all_preds, all_true = [], []

    with torch.no_grad():
        for bX, by in loader:
            bX, by = bX.to(device), by.to(device)

            preds = model(bX)
            mae_metric.update(preds, by)
            mse_metric.update(preds, by)

            all_preds.append(preds.cpu())
            all_true.append(by.cpu())

    ss_res = ((all_true - all_preds) ** 2).sum()
    ss_tot = ((all_true - all_true.mean()) ** 2).sum()
    r2 = (1 - ss_res / (ss_tot + 1e-8)).item()

    mae_val = mae_metric.compute().item()
    rmse_val = torch.sqrt(mse_metric.compute()).item()

    return {
        'mae' : mae_val * y_std, 'rmse' : rmse_val * y_std, 'r2' : r2
    }

In [17]:
# Evaluation During Training
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchmetrics

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

n = 2000
X = torch.randn(n, 10)
y = ((X[:, 0] + X[:, 1] * 0.5 - X[:, 2] * 0.3) > 0).float()

X_train, X_val = X[:1600], X[1600:]
y_train, y_val = y[:1600], y[1600:]

train_loader = DataLoader(TensorDataset(X_train, y_train.unsqueeze(1)), batch_size = 64, shuffle = True)
val_loader = DataLoader(TensorDataset(X_val, y_val.unsqueeze(1)), batch_size = 128, shuffle = False)

model = nn.Sequential(
    nn.Linear(10, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1)).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

train_metrics = {
    'acc' : torchmetrics.Accuracy(task = 'binary').to(device),
    'f1' : torchmetrics.F1Score(task = 'binary').to(device)
}
val_metrics = {
    'acc' :  torchmetrics.Accuracy(task='binary').to(device),
    'prec' : torchmetrics.Precision(task='binary').to(device),
    'rec' :  torchmetrics.Recall(task='binary').to(device),
    'f1' :   torchmetrics.F1Score(task='binary').to(device),
    'auc' :  torchmetrics.AUROC(task='binary').to(device),
}

history = []
best_val_f1 = 0.0
best_val_loss = float('inf')

for epoch in range(30):
    model.train()
    for m in train_metrics.values(): 
        m.reset()
    train_loss, train_n = 0.0, 0

    for bX, by in train_loader:
        bX, by = bX.to(device), by.to(device)

        optimizer.zero_grad()
        logits = model(bX)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()

        probs = torch.sigmoid(logits).squeeze(1)
        lbls = by.squeeze(1).long()
        
        for m in train_metrics.values():
            m.update(probs, lbls)

        train_loss += loss.item() * bX.size(0)
        train_n += bX.size(0)

    model.eval()
    for m in val_metrics.values():
        m.reset()
    val_loss, val_n = 0.0, 0

    with torch.no_grad():
        for bX, by in val_loader:
            bX, by = bX.to(device), by.to(device)

            logits = model(bX)
            loss = criterion(logits, by)

            probs = (torch.sigmoid(logits).squeeze(1))
            lbls = by.squeeze(1).long()
            for m in val_metrics.values():
                m.update(probs, lbls)

            val_loss += loss.item() * bX.size(0)
            val_n += bX.size(0)

    t_loss = train_loss / train_n
    v_loss = val_loss / val_n
    t_acc = train_metrics['acc'].compute().item()
    v_acc = val_metrics['acc'].compute().item()
    v_prec = val_metrics['prec'].compute().item()
    v_rec = val_metrics['rec'].compute().item()
    v_f1 = val_metrics['f1'].compute().item()
    v_auc = val_metrics['auc'].compute().item()

    epoch_data = {
        'epoch' : epoch + 1, 'train_loss' : t_loss, 'val_loss' : v_loss, 'train_acc' : t_acc, 
        'val_acc' : v_acc, 'v_precision' : v_prec, 'val_recall' : v_rec,
        'val_f1' : v_f1, 'val_auc' : v_auc
    }
    history.append(epoch_data)

    if v_f1 > best_val_f1:
        best_val_f1 = v_f1
        torch.save(model.state_dict(), 'best_model_by_f1.pth')

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:>3} | Loss {t_loss:.4f}/{v_loss:.4f} | "
              f"Acc {t_acc:.3f}/{v_acc:.3f} | F1 {v_f1:.3f} | AUC {v_auc:.3f} | P {v_prec:.3f} R {v_rec:.3f}")

print(f'\nBest Validation F1: {best_val_f1:.4f}')

Epoch  10 | Loss 0.0603/0.0622 | Acc 0.991/0.985 | F1 0.986 | AUC 0.999 | P 0.986 R 0.986
Epoch  20 | Loss 0.0237/0.0384 | Acc 0.999/0.988 | F1 0.988 | AUC 0.999 | P 0.986 R 0.991
Epoch  30 | Loss 0.0124/0.0329 | Acc 0.999/0.990 | F1 0.991 | AUC 1.000 | P 0.986 R 0.995

Best Validation F1: 0.9906


In [21]:
# Mini Exercise = Training and Evaluation Pipeline for churn classification

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import torchmetrics
import numpy as np

torch.manual_seed(42)
np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 1. Creating Imbalanced binary dataset: 90% clsss 0, 10% class 1
n = 3000
X = torch.randn(n, 10)

signal = X[:, 0] * 1.5 + X[:, 1] * 1.0 - 2.5  # biased towards negative
y = (signal > 0).float()
print(f'Class Imbalance: {y.mean():.3f} positive ({y.sum():.0f}/{n})')

X_train, X_val = X[:2400], X[2400:]
y_train, y_val = y[:2400], y[2400:]

train_loader = DataLoader(TensorDataset(X_train, y_train.unsqueeze(1)), batch_size = 64, shuffle = True)
val_loader = DataLoader(TensorDataset(X_val, y_val.unsqueeze(1)), batch_size = 128, shuffle = False)

# 2. Model
model = nn.Sequential(
    nn.Linear(10, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1)).to(device)

# 3. Loss with class weighting for imbalance: pos_weight = n_negative / n_positive = 9
pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight = pos_weight)
optimizer = optim.Adam(model.parameters(), lr = 0.001)

# 4. Metric Objects
def make_metrics(device):
    return {
        'acc':  torchmetrics.Accuracy(task='binary').to(device),
        'prec': torchmetrics.Precision(task='binary').to(device),
        'rec':  torchmetrics.Recall(task='binary').to(device),
        'f1':   torchmetrics.F1Score(task='binary').to(device),
        'auc':  torchmetrics.AUROC(task='binary').to(device),
    }
val_m = make_metrics(device)

# 5. Training loop with full metric reporting
best_val_f1 = 0.0
for epoch in range(50):
    model.train()
    train_loss, n_train = 0.0, 0
    for bX, by in train_loader:
        bX, by = bX.to(device), by.to(device)
        
        optimizer.zero_grad()
        logits = model(bX)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * bX.size(0)
        n_train += bX.size(0)

    model.eval()
    for m in val_m.values():
        m.reset()
    val_loss, n_val = 0.0, 0

    with torch.no_grad():
        for bX, by in train_loader:
            bX, by = bX.to(device), by.to(device)
            
            logits = model(bX)
            loss = criterion(logits, by)
            probs = torch.sigmoid(logits).squeeze(1)
            lbls = by.squeeze(1).long()

            for m in val_m.values():
                m.update(probs, lbls)
            val_loss += loss.item() * bX.size(0)
            n_val += bX.size(0)

    v_f1 = val_m['f1'].compute().item()
    v_auc = val_m['auc'].compute().item()

    if v_f1 > best_val_f1:
        best_val_f1 = v_f1
        torch.save(model.state_dict(), 'best_churn.pth')

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:>3} | Loss {train_loss/n_train:.4f}/{val_loss/n_val:.4f} | "
              f"Acc {val_m['acc'].compute():.3f} | P {val_m['prec'].compute():.3f} | "
              f"R {val_m['rec'].compute():.3f} | F1 {v_f1:.3f} | AUC {v_auc:.3f}")

# 6. Final Evaluation with confusion matrix
model.load_state_dict(torch.load('best_churn.pth'))
model.eval()

all_probs, all_labels = [], []
with torch.no_grad():
    for bX, by in val_loader:
        bX, by = bX.to(device), by.to(device)

        probs = torch.sigmoid(model(bX)).squeeze(1).cpu()
        all_probs.append(probs)
        all_labels.append(by.squeeze(1))

all_probs = torch.cat(all_probs).to(device)
all_labels = torch.cat(all_labels).long().to(device)
all_preds = (all_probs >= 0.5).long().to(device)

TP = ((all_preds == 1) & (all_labels == 1)).sum().item()
TN = ((all_preds == 0) & (all_labels == 0)).sum().item()
FP = ((all_preds == 1) & (all_labels == 0)).sum().item()
FN = ((all_preds == 0) & (all_labels == 1)).sum().item()

print(f"\n── Final Confusion Matrix ──")
print(f"              Pred 0   Pred 1")
print(f"  Actual 0  [  {TN:>4}    {FP:>4}  ]")
print(f"  Actual 1  [  {FN:>4}    {TP:>4}  ]")
print(f"\n  Precision: {TP/(TP+FP+1e-8):.3f}")
print(f"  Recall:    {TP/(TP+FN+1e-8):.3f}")
print(f"  F1:        {2*TP/(2*TP+FP+FN+1e-8):.3f}")
print(f"\nBest F1 achieved: {best_val_f1:.4f}")

Class Imbalance: 0.092 positive (275/3000)
Epoch  10 | Loss 0.1249/0.1152 | Acc 0.971 | P 0.754 | R 1.000 | F1 0.859 | AUC 0.998
Epoch  20 | Loss 0.0591/0.0528 | Acc 0.985 | P 0.856 | R 1.000 | F1 0.922 | AUC 1.000
Epoch  30 | Loss 0.0309/0.0292 | Acc 0.991 | P 0.907 | R 1.000 | F1 0.951 | AUC 1.000
Epoch  40 | Loss 0.0195/0.0174 | Acc 0.996 | P 0.960 | R 1.000 | F1 0.979 | AUC 1.000
Epoch  50 | Loss 0.0116/0.0112 | Acc 0.998 | P 0.973 | R 1.000 | F1 0.986 | AUC 1.000

── Final Confusion Matrix ──
              Pred 0   Pred 1
  Actual 0  [   533       6  ]
  Actual 1  [     0      61  ]

  Precision: 0.910
  Recall:    1.000
  F1:        0.953

Best F1 achieved: 0.9885
